In [ ]:
# Deploy Manny: log → register in Unity Catalog → staging endpoint → eval gate → production endpoint (agents.deploy)
# agents.deploy creates the Model Serving endpoint with MLflow tracing, an inference table (request/response
# payloads in UC Delta) and the review app. Lakebase and Geoapify credentials reach the endpoint as
# environment variables backed by the `maintops` secret scope — never in code.

In [ ]:
%pip install -q -U "mlflow>=3.1" databricks-agents databricks-sdk openai "psycopg[binary]>=3.1" psycopg_pool requests

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))   # repo root: manny.py imports maintops_core
import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex

UC_MODEL      = "bootcamp_students.maintops.manny"
ENDPOINT_NAME = "maintops-manny"
STAGING_ENDPOINT = "maintops-manny-staging"     # temporary: the new version is evaluated here before production
LLM_ENDPOINT  = "databricks-claude-sonnet-4-6"
FAQ_INDEX     = "bootcamp_students.maintops.faq_index"

dbutils.widgets.text("eval_repeats", "3")          # runs per eval scenario; any failed check blocks the deploy
EVAL_REPEATS = int(dbutils.widgets.get("eval_repeats"))
dbutils.widgets.text("eval_workers", "2")          # parallel conversations; more overloads Lakebase (CU) and the LLM rate limit
EVAL_WORKERS = int(dbutils.widgets.get("eval_workers"))

w = WorkspaceClient()
me = w.current_user.me().user_name
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{me}/maintops_manny")

# Needed while logging (the input example is run once); the endpoint gets them from the secret scope
os.environ["LAKEBASE_PG_URL"]  = dbutils.secrets.get("maintops", "lakebase_pg_url")
os.environ["GEOAPIFY_API_KEY"] = dbutils.secrets.get("maintops", "geoapify_api_key")

ENV_VARS = {
    "LAKEBASE_PG_URL":  "{{secrets/maintops/lakebase_pg_url}}",
    "GEOAPIFY_API_KEY": "{{secrets/maintops/geoapify_api_key}}",
    "MANNY_LLM_ENDPOINT": LLM_ENDPOINT,
}


def wait_ready(name: str, minutes: int = 45) -> None:
    deadline = time.time() + minutes * 60
    while time.time() < deadline:
        ep = w.serving_endpoints.get(name)
        ready, updating = str(ep.state.ready), str(ep.state.config_update)
        print(f"{time.strftime('%H:%M:%S')} {name}: ready={ready} config_update={updating}")
        if ready.endswith("READY") and not updating.endswith("IN_PROGRESS"):
            return
        time.sleep(60)
    raise TimeoutError(f"{name} not ready after {minutes} minutes")

In [ ]:
input_example = {
    "input": [{"role": "user", "content": "How does MaintOps choose a handyman?"}],
    "custom_inputs": {"role": "visitor"},
}
with mlflow.start_run(run_name="manny"):
    info = mlflow.pyfunc.log_model(
        name="manny",
        python_model="manny.py",                      # models-from-code: agent/manny.py
        code_paths=["../maintops_core"],              # shared services packaged with the model
        resources=[DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
                   DatabricksVectorSearchIndex(index_name=FAQ_INDEX)],
        pip_requirements=["mlflow>=3.1", "databricks-sdk", "openai", "httpx", "psycopg[binary]>=3.1",
                          "psycopg_pool", "requests"],
        input_example=input_example,
    )
print(info.model_uri)

In [ ]:
registered = mlflow.register_model(info.model_uri, UC_MODEL)
print(f"Registered {UC_MODEL} version {registered.version}")

In [ ]:
# The new version on a temporary staging endpoint, configured like production
from databricks import agents

agents.deploy(UC_MODEL, registered.version, endpoint_name=STAGING_ENDPOINT, scale_to_zero=False,
              environment_vars=ENV_VARS, tags={"project": "maintops", "stage": "staging"})
wait_ready(STAGING_ENDPOINT)


def wait_answering(name: str, minutes: int = 10) -> None:
    """READY is reported before every replica serves requests: send a harmless visitor message until one is
    answered, so the first eval conversations do not hit "endpoint does not exist"."""
    deadline = time.time() + minutes * 60
    while True:
        try:
            w.api_client.do("POST", f"/serving-endpoints/{name}/invocations",
                            body={"input": [{"role": "user", "content": "Hi"}], "custom_inputs": {"role": "visitor"}})
            print(f"{time.strftime('%H:%M:%S')} {name} answers")
            return
        except Exception as exc:  # noqa: BLE001 — retried until the deadline
            if time.time() > deadline:
                raise TimeoutError(f"{name} does not answer after {minutes} minutes: {exc}") from exc
            print(f"{time.strftime('%H:%M:%S')} {name} not answering yet: {str(exc)[:120]}")
            time.sleep(30)


wait_answering(STAGING_ENDPOINT)

In [ ]:
# Release gate: the full evaluation, eval/manny_eval.py, against the staging endpoint — real conversations on dedicated
# test accounts, every write, tool argument and figure checked against Lakebase. One failed check in any run stops
# the job here and production keeps the current version. The staging endpoint is deleted either way.
sys.path.insert(0, os.path.abspath("../eval"))
import manny_eval

try:
    report = manny_eval.evaluate(manny_eval.EndpointBackend(STAGING_ENDPOINT), repeats=EVAL_REPEATS, workers=EVAL_WORKERS,
                                 judges=False)   # judges share the LLM token budget; run them separately
finally:
    w.serving_endpoints.delete(STAGING_ENDPOINT)
if not report["passed"]:
    raise RuntimeError(f"Eval gate FAILED for version {registered.version}: {report['passed_runs']}/{report['runs']} "
                       f"runs passed, {report['checks_failed']} failed checks (MLflow run {report['mlflow_run_id']}). "
                       "Not deploying.")

In [ ]:
deployment = agents.deploy(
    UC_MODEL, registered.version,
    endpoint_name=ENDPOINT_NAME,
    scale_to_zero=False,                              # the web app needs Manny without cold starts
    environment_vars=ENV_VARS,
    tags={"project": "maintops"},
)
print(deployment)

In [ ]:
# Wait until the new version serves, then smoke-test it
wait_ready(ENDPOINT_NAME)

resp = w.api_client.do("POST", f"/serving-endpoints/{ENDPOINT_NAME}/invocations", body={
    "input": [{"role": "user", "content": "Can I choose the handyman myself?"}],
    "custom_inputs": {"role": "visitor"}})
print(resp["output"][0]["content"][0]["text"])

In [ ]:
# agents.deploy keeps every earlier version on the endpoint, each with its own always-on replica.
# Serve only the new version.
ep = w.api_client.do("GET", f"/api/2.0/serving-endpoints/{ENDPOINT_NAME}")
entities = ep["config"]["served_entities"]
if len(entities) > 1:
    new = next(e for e in entities if e["entity_version"] == str(registered.version))
    keep = {k: new[k] for k in ("name", "entity_name", "entity_version", "environment_vars", "scale_to_zero_enabled",
                                "workload_size", "workload_type") if k in new}
    w.api_client.do("PUT", f"/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config", body={
        "served_entities": [keep],
        "traffic_config": {"routes": [{"served_model_name": keep["name"], "traffic_percentage": 100}]}})
    while str(w.serving_endpoints.get(ENDPOINT_NAME).state.config_update).endswith("IN_PROGRESS"):
        time.sleep(30)
    print(f"Removed versions {[e['entity_version'] for e in entities if e is not new]}; serving only {registered.version}")

# Mark the version that passed the gate and now serves production
from mlflow import MlflowClient
MlflowClient().set_registered_model_alias(UC_MODEL, "production", registered.version)
print(f"{UC_MODEL}@production = version {registered.version}")